# 04 — Audit Kompatibilitas ECGTransForm (feasibility test)

Notebook ini **hanya** compatibility audit. Pertanyaan tunggalnya:

> Apakah implementasi ECGTransForm dapat dipetakan ke dataset MIT-BIH 87.554 / 21.892 milik proyek ini dengan perubahan minimal, sehingga layak dipakai sebagai metode pembanding terhadap ResNet18 + GAF?

| Dilakukan | TIDAK dilakukan |
|---|---|
| Membaca source resmi + paper; cek shape, label, range; 1 adapter (notebook-only); **1× forward, 1× loss, 1× backward, 1× `optimizer.step()`** pada model berbobot acak (batch = 8) | Full training / epoch, tuning, evaluasi performa, perubahan Week 1–3, perubahan test set, klaim "lebih baik" atau "replikasi berhasil" |

**Label bukti** — `FACT`: terlihat langsung dari source / paper / pengukuran. `INFERENCE`: kesimpulan teknis dari FACT. `UNKNOWN`: tidak dapat dipastikan dari sumber yang diperiksa.

**Jaminan keamanan**
- `X_test.npy` / `y_test.npy` dibuka read-only dan di-hash sebelum & sesudah audit (Check 10).
- File repo ECGTransForm di-hash sebelum & sesudah audit, dan `git status` harus bersih (Check 11). Repo di-clone ke luar Drive.
- Tidak ada checkpoint / model yang disimpan. Satu-satunya file yang ditulis ke Drive: `outputs/results/ecgtransform_audit.json` (file baru, tidak menimpa apa pun).
- Angka hasil paper tidak dipakai sebagai hasil eksperimen di notebook ini.

# 0. Environment & Konfigurasi

In [1]:
import sys
sys.dont_write_bytecode = True  # jangan menulis __pycache__ ke dalam repo ECGTransForm (Check 11)

import hashlib, json, os, re, subprocess, tempfile, time
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

pd.set_option("display.max_colwidth", 240)
pd.set_option("display.width", 240)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---- Path. Environment variable hanya untuk menjalankan di luar Colab; default = path proyek.
BASE_DIR = os.environ.get("ECG_BASE_DIR", "/content/drive/MyDrive/Projek PSD/ECG_Multimodal_Fusion")
FEATURES_DIR = os.environ.get("ECG_FEATURES_DIR", os.path.join(BASE_DIR, "features"))
RESULTS_DIR = os.environ.get("ECG_RESULTS_DIR", os.path.join(BASE_DIR, "outputs", "results"))
WORK_DIR = os.environ.get("ECG_WORK_DIR", "/content" if IN_COLAB else tempfile.gettempdir())

REPO_URL = "https://github.com/emadeldeen24/ECGTransForm.git"
REPO_COMMIT = "2f5c28c088ff65681649e3c267c0894336ec412a"  # commit yang diaudit
REPO_DIR = os.path.join(WORK_DIR, "ECGTransForm")           # di luar Drive dan di luar folder Week 1-3

PATHS = {n: os.path.join(FEATURES_DIR, n)
         for n in ("X_train_smote.npy", "y_train_smote.npy", "X_test.npy", "y_test.npy")}

# ---- Target proyek (dari brief) dan referensi paper
N_TRAIN_RAW, N_TEST, SEQ_LEN, N_CLASSES = 87554, 21892, 187, 5
CLASS_NAMES = ["N", "S", "V", "F", "Q"]
EXPECTED_TEST = {0: 18118, 1: 556, 2: 1448, 3: 162, 4: 1608}
EXPECTED_TRAIN_RAW = {0: 72471, 1: 2223, 2: 5788, 3: 641, 4: 6431}  # Table 1 paper ECGTransForm (referensi)
BASELINE_CLASS_WEIGHTS = [1, 2, 1, 2, 1]  # 01_training.ipynb -> build_baseline()
SEED = 42
AUDIT_BATCH = 8
SAVE_JSON = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def sha256_array(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()


def dist(y):
    u, c = np.unique(np.asarray(y).astype(np.int64), return_counts=True)
    return {int(k): int(v) for k, v in zip(u, c)}


AUDIT = {}         # no -> dict(audit, status, evidence, adaptation)
ADAPTATIONS = []   # dict(item, level, detail); level: "minor" | "protocol"
UNKNOWNS = []      # risiko residual yang tidak dapat dipastikan


def record(no, audit, status, evidence, adaptation="-"):
    AUDIT[no] = dict(audit=audit, status=status, evidence=evidence, adaptation=adaptation)
    print(f"[{status}] #{no} {audit}: {evidence}")


print("torch      :", torch.__version__)
print("DEVICE     :", DEVICE)
print("IN_COLAB   :", IN_COLAB)
print("FEATURES   :", FEATURES_DIR)
print("REPO_DIR   :", REPO_DIR)

missing = [n for n, p in PATHS.items() if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Artifact proyek tidak ditemukan: " + ", ".join(missing) + f"\nPeriksa FEATURES_DIR = {FEATURES_DIR}. "
        "Audit dihentikan agar tidak ada asumsi diam-diam."
    )
print("Semua artifact features/*.npy ditemukan.")

torch      : 2.14.1
DEVICE     : cpu
IN_COLAB   : False
FEATURES   : /mnt/user-data/uploads
REPO_DIR   : /home/claude/work/wd/ECGTransForm
Semua artifact features/*.npy ditemukan.


# A. Audit source resmi ECGTransForm

Repo di-clone ke `REPO_DIR` pada commit tetap, di-hash, lalu **tidak pernah diedit**. Modul `configs`, `dataloader`, `models` diimpor apa adanya. `trainer.py` dan `main.py` **tidak dijalankan** (itu akan memulai training 60 epoch).

In [2]:
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REPO_COMMIT], check=True)

HEAD = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert HEAD == REPO_COMMIT, f"HEAD {HEAD} != commit yang diaudit {REPO_COMMIT}"
print("Repo   :", REPO_URL)
print("Commit :", HEAD)


def repo_snapshot():
    snap = {}
    for root, dirs, files in os.walk(REPO_DIR):
        dirs[:] = [d for d in dirs if d != ".git"]
        for f in files:
            p = os.path.join(root, f)
            snap[os.path.relpath(p, REPO_DIR)] = sha256_file(p)
    return snap


REPO_BEFORE = repo_snapshot()   # untuk Check 11
print("File repo di-hash:", len(REPO_BEFORE))

sys.path.insert(0, REPO_DIR)
from configs.data_configs import mit              # noqa: E402  (kode asli repo, tidak diubah)
from configs.hparams import supervised            # noqa: E402
from dataloader import Load_Dataset, data_generator, get_class_weight  # noqa: E402
from models import ecgTransForm                   # noqa: E402

CFG = mit()
HP = supervised().train_params
print("\nconfig `mit`        :", vars(CFG))
print("hparams `supervised`:", HP)

Repo   : https://github.com/emadeldeen24/ECGTransForm.git
Commit : 2f5c28c088ff65681649e3c267c0894336ec412a
File repo di-hash: 12



config `mit`        : {'num_classes': 5, 'class_names': ['N', 'S', 'V', 'F', 'Q'], 'sequence_len': 186, 'input_channels': 1, 'kernel_size': 8, 'stride': 1, 'dropout': 0.2, 'mid_channels': 32, 'final_out_channels': 128, 'trans_dim': 25, 'num_heads': 5}
hparams `supervised`: {'num_epochs': 60, 'batch_size': 128, 'weight_decay': 0.0001, 'learning_rate': 0.001, 'feature_dim': 128}


In [3]:
def show(rel, start=1, end=None):
    with open(os.path.join(REPO_DIR, rel), newline=None) as f:
        lines = f.read().splitlines()
    for i, line in enumerate(lines[start - 1:end], start):
        print(f"{i:4d}  {line}")


for rel in ["configs/data_configs.py", "configs/hparams.py", "dataloader.py", "models.py"]:
    print("=" * 110, f"\n{rel}\n", "=" * 110, sep="")
    show(rel)

configs/data_configs.py
   1  def get_dataset_class(dataset_name):
   2      """Return the algorithm class with the given name."""
   3      if dataset_name not in globals():
   4          raise NotImplementedError("Dataset not found: {}".format(dataset_name))
   5      return globals()[dataset_name]
   6  
   7  
   8  class mit():
   9      def __init__(self):
  10          super(mit, self).__init__()
  11          # data parameters
  12          self.num_classes = 5
  13          self.class_names = ['N', 'S', 'V', 'F', 'Q']
  14          self.sequence_len = 186
  15  
  16          # model configs
  17          self.input_channels = 1
  18          self.kernel_size = 8
  19          self.stride = 1
  20          self.dropout = 0.2
  21  
  22          # features
  23          self.mid_channels = 32
  24          self.final_out_channels = 128
  25  
  26          # Transformer
  27          self.trans_dim = 25
  28          self.num_heads = 5
  29  
  30  
  31  class ptb():
  32    

In [4]:
def read_py(rel):
    with open(os.path.join(REPO_DIR, rel), newline=None) as f:
        return f.read()


PY_FILES = sorted(r for r in REPO_BEFORE if r.endswith(".py"))


def scan(pattern, files=PY_FILES):
    return [(rel, i, line.strip()) for rel in files
            for i, line in enumerate(read_py(rel).splitlines(), 1) if re.search(pattern, line, re.I)]


def show_hits(title, hits):
    print(title)
    if not hits:
        print("    (tidak ada)")
    for rel, i, line in hits:
        print(f"    {rel}:{i}: {line}")
    print()


hits_seq = scan(r"sequence_len")
show_hits("[1] Semua kemunculan `sequence_len` di seluruh .py repo:", hits_seq)
SEQ_LEN_USED_IN_CODE = any(not rel.startswith("configs/") for rel, _, _ in hits_seq)
print("    -> dipakai di luar file config?", SEQ_LEN_USED_IN_CODE, "\n")

used_cfg = sorted(set(re.findall(r"configs\.(\w+)", read_py("models.py"))))
unused_cfg = sorted(set(vars(CFG)) - set(used_cfg))
print("[2] Atribut config yang DIBACA models.py :", used_cfg)
print("    Atribut config yang TIDAK dibaca     :", unused_cfg, "\n")

show_hits("[3] Normalisasi / transform di kode repo:", scan(r"transforms|normaliz|scaler|standardi|minmax|zscore"))
show_hits("[4] R-peak / resampling / filter / denoising / SMOTE / oversampling di kode repo:",
          scan(r"peak|smote|resampl|denois|butter|savgol|filtfilt|np\.pad|F\.pad|oversampl|imblearn|sampler"))
show_hits("[5] Format file yang dibaca dataloader:", scan(r"torch\.load|\.pt\b", ["dataloader.py"]))
show_hits("[6] Protokol training/evaluasi di trainer.py (dibaca saja, TIDAK dijalankan):",
          scan(r"Adam|weight_decay|CrossEntropyLoss|best_f1|val_dl|test_dl|validation_best|test_last|save_checkpoint\(self", ["trainer.py"]))

[1] Semua kemunculan `sequence_len` di seluruh .py repo:
    configs/data_configs.py:14: self.sequence_len = 186
    configs/data_configs.py:37: self.sequence_len = 188

    -> dipakai di luar file config? False 

[2] Atribut config yang DIBACA models.py : ['dropout', 'final_out_channels', 'input_channels', 'mid_channels', 'num_classes', 'num_heads', 'stride', 'trans_dim']
    Atribut config yang TIDAK dibaca     : ['class_names', 'kernel_size', 'sequence_len'] 

[3] Normalisasi / transform di kode repo:
    dataloader.py:4: from torchvision import transforms
    utils.py:173: # Normalize the labels to [0, 1] for colormap

[4] R-peak / resampling / filter / denoising / SMOTE / oversampling di kode repo:
    (tidak ada)

[5] Format file yang dibaca dataloader:
    dataloader.py:50: train_dataset = torch.load(os.path.join(data_path, data_type, f"train.pt"))
    dataloader.py:51: val_dataset = torch.load(os.path.join(data_path, data_type, f"val.pt"))
    dataloader.py:52: test_dataset = t

### Ringkasan temuan source (commit `2f5c28c`)

| Sumber | Temuan | Label |
|---|---|---|
| `configs/data_configs.py` · class `mit` | `num_classes=5`, `class_names=['N','S','V','F','Q']`, `input_channels=1`, `sequence_len=186`, `trans_dim=25`, `num_heads=5`, `dropout=0.2`, `final_out_channels=128` | FACT |
| | `sequence_len` hanya didefinisikan, **tidak dibaca** oleh file `.py` mana pun (scan [1]) → bukan validasi / pemotongan input | FACT |
| | `kernel_size` di config juga tidak dibaca `models.py` (kernel di-hardcode di model) | FACT (scan [2]) |
| | Paper §4.2 menyebut dropout 0,5, sedangkan config memakai 0,2 → diskrepansi paper vs kode | FACT |
| `configs/hparams.py` | Satu set hparams untuk semua dataset (tidak ada konfigurasi khusus MIT-BIH): 60 epoch, batch 128, lr 1e-3, weight decay 1e-4, `feature_dim=128` | FACT |
| `dataloader.py` | `torch.load(<data_path>/<dataset>/{train,val,test}.pt)`; tiap file = dict `{"samples", "labels"}`; samples → float32, labels → long | FACT |
| | `__getitem__` memakai `x[i].squeeze(-1)` ⇒ agar batch berbentuk `(B,1,L)`, `samples` di file harus `(N,1,L)` | INFERENCE (dibuktikan di Check 5) |
| | **Wajib ada `val.pt`**; proyek ini tidak punya validation split | FACT |
| | Tidak ada normalisasi / transform (`transforms` hanya di-import); tidak ada R-peak, resampling, filter, SMOTE (scan [3], [4]) | FACT |
| | Bobot kelas dari label train: `w_i = max(1, log(n_max / n_i))` (Context-Aware Loss) | FACT |
| `models.py` | Conv1d multi-skala (k = 5, 9, 11, padding "same") → dirata-rata → 2 blok conv + MaxPool(2, 2, pad 1) → 3× blok SE-residual (CRM) → TransformerEncoder (d_model = 25, 5 head, 3 layer, dua arah via `flip`) → AdaptiveAvgPool1d → Linear(128, 5) | FACT |
| | `d_model = 25` menuntut panjang sumbu waktu setelah 3× pooling tepat 25; panjang input 186 **dan** 187 sama-sama menghasilkan 25 | INFERENCE → dibuktikan numerik di Check 1 & 6 |
| | `self.encoder_layer` terdaftar sebagai submodul tetapi tidak dipakai langsung (hanya disalin oleh `TransformerEncoder`) → parameternya tidak menerima gradien | INFERENCE → diverifikasi di Check 9 |
| `trainer.py` | Adam(lr 1e-3, wd 1e-4, betas 0,9 / 0,99); `CrossEntropyLoss(weight=CAL)`; checkpoint terbaik dipilih dari **F1 validasi**, test dievaluasi pada **epoch terakhir** | FACT (scan [6]) |
| Paper §4.1.3 | Preprocessing "mengikuti [37]" = Kachuee dkk. (sumber dataset Kaggle `shayanfazeli/heartbeat`): window 10 s → normalisasi amplitudo [0, 1] → kandidat R-peak (threshold 0,9) → median RR → segmen 1,2·T → zero padding ke panjang tetap | FACT |
| Paper Table 1 | MIT-BIH train 87.554 (72471 / 2223 / 5788 / 641 / 6431) dan test 21.892 (18118 / 556 / 1448 / 162 / 1608), **identik** dengan split proyek | FACT |
| Paper §4.2 | Split 80/20, lalu 20% dari train untuk validasi; 3 seed; 60 epoch | FACT |
| Paper | Panjang tetap hasil zero padding **tidak disebutkan**; ada / tidaknya denoising, filtering, resampling 360 → 125 Hz **tidak disebutkan** | UNKNOWN |
| File `.pt` milik penulis (Google Drive di README) | Tidak diunduh / diperiksa audit ini → shape & dtype aktual file penulis tidak diverifikasi | UNKNOWN |

# B. Dataset proyek

`X_train_smote.npy` adalah **hasil SMOTE** (Week 1), bukan raw train set. Test set (`X_test.npy`, `y_test.npy`) dibuka **read-only** dan di-hash sekarang untuk dibandingkan lagi di Check 10.

In [5]:
# ---- Snapshot test set SEBELUM audit (dibandingkan lagi di Check 10)
TEST_BEFORE = {
    "x_file_sha256": sha256_file(PATHS["X_test.npy"]),
    "y_file_sha256": sha256_file(PATHS["y_test.npy"]),
}

X_train_smote = np.load(PATHS["X_train_smote.npy"])
y_train_smote = np.load(PATHS["y_train_smote.npy"])
X_test = np.load(PATHS["X_test.npy"])
y_test = np.load(PATHS["y_test.npy"])
for a in (X_train_smote, y_train_smote, X_test, y_test):
    a.setflags(write=False)  # read-only: audit tidak dapat menulis ke array ini

TEST_BEFORE.update(
    x_shape=X_test.shape, y_shape=y_test.shape, dist=dist(y_test),
    x_array_sha256=sha256_array(X_test), y_array_sha256=sha256_array(y_test),
)
print("Test set (sebelum audit):", {k: (v if "sha256" not in k else v[:16] + "...") for k, v in TEST_BEFORE.items()})

# ---- Apakah raw train / dataset asli tersedia di artifact?
RAW_CSV_CANDIDATES = [os.path.join(d, "mitbih_train.csv") for d in
                      (FEATURES_DIR, BASE_DIR, os.path.join(BASE_DIR, "data"),
                       os.path.join(BASE_DIR, "raw"), os.path.join(BASE_DIR, "dataset"))]
RAW_CSV = next((p for p in RAW_CSV_CANDIDATES if os.path.exists(p)), None)
print("\nmitbih_train.csv (raw Kaggle) di artifact:", RAW_CSV or "TIDAK DITEMUKAN")

Test set (sebelum audit): {'x_file_sha256': 'b9f0adee448c71e2...', 'y_file_sha256': '101e508029e30db3...', 'x_shape': (21892, 187), 'y_shape': (21892,), 'dist': {0: 18118, 1: 556, 2: 1448, 3: 162, 4: 1608}, 'x_array_sha256': 'd7230d57a048f158...', 'y_array_sha256': '6e150f8e0e3915aa...'}

mitbih_train.csv (raw Kaggle) di artifact: TIDAK DITEMUKAN


## Check 1 — Shape, dtype, jumlah kelas, dan *original shape* train

In [6]:
rows = []
for name, a in (("X_train_smote", X_train_smote), ("y_train_smote", y_train_smote), ("X_test", X_test), ("y_test", y_test)):
    rows.append({"array": name, "shape": a.shape, "dtype": str(a.dtype),
                 "n_sample": a.shape[0], "n_fitur": a.shape[1] if a.ndim == 2 else "-"})
display(pd.DataFrame(rows))

# ---- Original shape train: hanya tersedia sebagai data SMOTE
print(f"\nX_train_smote punya {len(X_train_smote):,} baris != {N_TRAIN_RAW:,} (target train mentah) -> ini DATA HASIL SMOTE, bukan raw train.")
X_raw, y_raw = X_train_smote[:N_TRAIN_RAW], y_train_smote[:N_TRAIN_RAW].astype(np.int64)
X_syn, y_syn = X_train_smote[N_TRAIN_RAW:], y_train_smote[N_TRAIN_RAW:].astype(np.int64)
dist_raw, dist_syn, dist_smote = dist(y_raw), dist(y_syn), dist(y_train_smote)

print("\nDistribusi X_train_smote[:87554]  :", dist_raw)
print("Referensi Table 1 paper (train asli):", EXPECTED_TRAIN_RAW)
print("Distribusi sisa (sintetik)         :", dist_syn)
RAW_SLICE_MATCHES_TABLE1 = dist_raw == EXPECTED_TRAIN_RAW
print("\n87554 baris pertama == distribusi train asli?          ", RAW_SLICE_MATCHES_TABLE1)
print("Sisa baris hanya berisi kelas minoritas (tanpa kelas 0)?", 0 not in dist_syn)

RAW_CSV_STATUS = "tidak tersedia"
if RAW_CSV:
    raw = pd.read_csv(RAW_CSV, header=None).values
    same_X = raw.shape == (N_TRAIN_RAW, SEQ_LEN + 1) and np.allclose(raw[:, :SEQ_LEN].astype(np.float32), X_raw, atol=1e-6)
    same_y = raw.shape[0] == N_TRAIN_RAW and np.array_equal(raw[:, SEQ_LEN].astype(np.int64), y_raw)
    RAW_CSV_STATUS = f"dibandingkan: X identik={same_X}, y identik={same_y}"
    print("\nPerbandingan dengan raw CSV:", RAW_CSV_STATUS)

print(f"""
KESIMPULAN (original shape train):
  FACT      : artifact hanya memuat train hasil SMOTE ({len(X_train_smote):,} baris). Raw CSV: {RAW_CSV_STATUS}.
  FACT      : 01_training.ipynb (Section 2) menyatakan sampel sintetik SMOTE berada di ujung array.
  INFERENCE : train mentah = X_train_smote[:{N_TRAIN_RAW}] karena distribusi kelasnya sama persis dengan Table 1 paper
              (72471 / 2223 / 5788 / 641 / 6431) dan sisanya hanya kelas minoritas.
  UNKNOWN   : bahwa 87.554 baris itu identik bit-per-bit dengan mitbih_train.csv Kaggle
              ({'terverifikasi' if RAW_CSV else 'file CSV tidak ada di artifact'}).
""")
if not (RAW_CSV and same_X and same_y):
    UNKNOWNS.append("Identitas bit-per-bit X_train_smote[:87554] dengan mitbih_train.csv Kaggle tidak dapat diverifikasi (CSV tidak ada di artifact).")

,array,shape,dtype,n_sample,n_fitur
0,X_train_smote,"(152471, 187)",float32,152471,187
1,y_train_smote,"(152471,)",float64,152471,-
2,X_test,"(21892, 187)",float32,21892,187
3,y_test,"(21892,)",int32,21892,-



X_train_smote punya 152,471 baris != 87,554 (target train mentah) -> ini DATA HASIL SMOTE, bukan raw train.

Distribusi X_train_smote[:87554]  : {0: 72471, 1: 2223, 2: 5788, 3: 641, 4: 6431}
Referensi Table 1 paper (train asli): {0: 72471, 1: 2223, 2: 5788, 3: 641, 4: 6431}
Distribusi sisa (sintetik)         : {1: 27777, 2: 14212, 3: 19359, 4: 3569}

87554 baris pertama == distribusi train asli?           True
Sisa baris hanya berisi kelas minoritas (tanpa kelas 0)? True

KESIMPULAN (original shape train):
  FACT      : artifact hanya memuat train hasil SMOTE (152,471 baris). Raw CSV: tidak tersedia.
  FACT      : 01_training.ipynb (Section 2) menyatakan sampel sintetik SMOTE berada di ujung array.
  INFERENCE : train mentah = X_train_smote[:87554] karena distribusi kelasnya sama persis dengan Table 1 paper
              (72471 / 2223 / 5788 / 641 / 6431) dan sisanya hanya kelas minoritas.
  UNKNOWN   : bahwa 87.554 baris itu identik bit-per-bit dengan mitbih_train.csv Kaggle
        

In [7]:
# ---- Ketergantungan panjang input: apakah 186 / 187 kompatibel dengan model? (model ASLI, bobot acak)
TRACE_MODULES = ["mp", "conv_block2", "conv_block3", "crm", "transformer_encoder", "aap", "clf"]


def trace(net, L, batch=2):
    """Jalankan 1 forward (eval, no_grad) dan catat shape keluaran tiap tahap. Tidak ada training."""
    shapes = {}

    def make_hook(name):
        def hook(module, inputs, output):   # hanya mencatat shape; WAJIB mengembalikan None (agar output modul tidak diganti)
            shapes.setdefault(name, tuple(output.shape))
        return hook

    hooks = [getattr(net, n).register_forward_hook(make_hook(n)) for n in TRACE_MODULES]
    was_training = net.training
    net.eval()
    try:
        with torch.no_grad():
            out = net(torch.randn(batch, CFG.input_channels, L, device=next(net.parameters()).device))
        return {"ok": True, "shapes": shapes, "out": tuple(out.shape), "final_len": shapes["crm"][-1], "error": ""}
    except Exception as e:  # noqa: BLE001  (kegagalan adalah data audit)
        return {"ok": False, "shapes": shapes, "out": None, "final_len": shapes.get("crm", (None,))[-1],
                "error": f"{type(e).__name__}: {str(e).splitlines()[0][:140]}"}
    finally:
        for h in hooks:
            h.remove()
        net.train(was_training)


torch.manual_seed(SEED)
_probe = ecgTransForm(configs=CFG, hparams=HP)
scan_rows = [{"L": L, **{k: v for k, v in trace(_probe, L).items() if k in ("ok", "final_len", "out", "error")}}
             for L in range(150, 231)]
scan_df = pd.DataFrame(scan_rows)
ok_L = scan_df.loc[scan_df.ok, "L"].tolist()
contiguous = ok_L == list(range(min(ok_L), max(ok_L) + 1))

print(f"trans_dim (d_model Transformer) = {CFG.trans_dim}")
print(f"Panjang input yang dapat diproses model (diuji L = 150..230): {min(ok_L)}..{max(ok_L)}" if contiguous else ok_L)
display(scan_df[scan_df.L.isin([180, 181, 185, 186, 187, 188, 189])].reset_index(drop=True))

T186, T187 = trace(_probe, 186), trace(_probe, 187)
SEQ_COMPATIBLE = bool(T186["ok"] and T187["ok"] and T186["final_len"] == T187["final_len"] == CFG.trans_dim)
print(f"\nL=186 -> panjang akhir {T186['final_len']} | L=187 -> panjang akhir {T187['final_len']} | trans_dim = {CFG.trans_dim}")
print("186 dan 187 menghasilkan feature map akhir yang sama dan kompatibel dengan trans_dim?", SEQ_COMPATIBLE)

last_col_zero_test = float((X_test[:, -1] == 0).mean())
last_col_zero_raw = float((X_raw[:, -1] == 0).mean())
print(f"\nKolom ke-187 bernilai 0 pada {last_col_zero_test:.1%} sampel test dan {last_col_zero_raw:.1%} sampel train-mentah (zero padding).")
print("INFERENCE: selisih 186 vs 187 pada config hampir tidak berdampak pada isi sinyal. "
      "UNKNOWN: apakah penulis memotong titik ke-187 atau memakai segmentasi lain.")
UNKNOWNS.append("Asal panjang 186 pada config penulis (dipotong dari 187 / segmentasi berbeda) tidak dapat dipastikan; file .pt penulis tidak diperiksa.")

/home/claude/work/wd/ECGTransForm/models.py:41: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=3)


trans_dim (d_model Transformer) = 25
Panjang input yang dapat diproses model (diuji L = 150..230): 180..187


,L,ok,out,final_len,error
0,180,True,"(2, 5)",25,
1,181,True,"(2, 5)",25,
2,185,True,"(2, 5)",25,
3,186,True,"(2, 5)",25,
4,187,True,"(2, 5)",25,
5,188,False,None,26,"AssertionError: was expecting embedding dimension of 25, but got 26"
6,189,False,None,26,"AssertionError: was expecting embedding dimension of 25, but got 26"



L=186 -> panjang akhir 25 | L=187 -> panjang akhir 25 | trans_dim = 25
186 dan 187 menghasilkan feature map akhir yang sama dan kompatibel dengan trans_dim? True

Kolom ke-187 bernilai 0 pada 99.1% sampel test dan 99.1% sampel train-mentah (zero padding).
INFERENCE: selisih 186 vs 187 pada config hampir tidak berdampak pada isi sinyal. UNKNOWN: apakah penulis memotong titik ke-187 atau memakai segmentasi lain.


## Check 2 — Label mapping

In [8]:
y_train_is_int = bool(np.all(y_train_smote == np.round(y_train_smote)))
classes_train = sorted(dist(y_train_smote))
classes_test = sorted(dist(y_test))
dist_test = dist(y_test)

mapping = pd.DataFrame({
    "id label": range(N_CLASSES),
    "repo: class_names[id]": CFG.class_names,
    "proyek: CLASS_NAMES[id]": CLASS_NAMES,
    "n test": [dist_test.get(i) for i in range(N_CLASSES)],
    "n test (target)": [EXPECTED_TEST[i] for i in range(N_CLASSES)],
})
mapping["cocok"] = (mapping["repo: class_names[id]"] == mapping["proyek: CLASS_NAMES[id]"]) & (mapping["n test"] == mapping["n test (target)"])
display(mapping)

LABEL_OK = (
    list(CFG.class_names) == CLASS_NAMES and CFG.num_classes == N_CLASSES
    and classes_train == classes_test == list(range(N_CLASSES))
    and y_train_is_int and dist_test == EXPECTED_TEST
)
print("y_train_smote berisi nilai bulat?        :", y_train_is_int, f"(dtype {y_train_smote.dtype})")
print("Kelas train / test                        :", classes_train, "/", classes_test)
print("Distribusi test == target brief           :", dist_test == EXPECTED_TEST)
print("num_classes repo == 5, urutan nama sama?  :", CFG.num_classes == N_CLASSES, list(CFG.class_names) == CLASS_NAMES)
print("""
FACT      : repo tidak meng-encode ulang label; dataloader hanya `.long()`. trainer memakai `class_names` sebagai
            target_names classification_report, jadi id i <-> class_names[i].
INFERENCE : mapping repo = proyek (0=N, 1=S, 2=V, 3=F, 4=Q) -> label TIDAK diubah.
Adaptasi  : hanya dtype (y_train float64 / y_test int32 -> int64), tanpa mengubah nilai.""")
ADAPTATIONS.append(dict(item="Cast label ke int64", level="minor",
                        detail="y_train_smote float64 dan y_test int32 -> int64; nilai label tidak berubah (Load_Dataset juga memanggil .long())."))
record(2, "Label", "PASS" if LABEL_OK else "FAIL",
       f"repo class_names={list(CFG.class_names)} == proyek; num_classes=5; test dist == target {dist_test}",
       "Tidak ada remapping; hanya cast dtype ke int64")

,id label,repo: class_names[id],proyek: CLASS_NAMES[id],n test,n test (target),cocok
0,0,N,N,18118,18118,True
1,1,S,S,556,556,True
2,2,V,V,1448,1448,True
3,3,F,F,162,162,True
4,4,Q,Q,1608,1608,True


y_train_smote berisi nilai bulat?        : True (dtype float64)
Kelas train / test                        : [0, 1, 2, 3, 4] / [0, 1, 2, 3, 4]
Distribusi test == target brief           : True
num_classes repo == 5, urutan nama sama?  : True True

FACT      : repo tidak meng-encode ulang label; dataloader hanya `.long()`. trainer memakai `class_names` sebagai
            target_names classification_report, jadi id i <-> class_names[i].
INFERENCE : mapping repo = proyek (0=N, 1=S, 2=V, 3=F, 4=Q) -> label TIDAK diubah.
Adaptasi  : hanya dtype (y_train float64 / y_test int32 -> int64), tanpa mengubah nilai.
[PASS] #2 Label: repo class_names=['N', 'S', 'V', 'F', 'Q'] == proyek; num_classes=5; test dist == target {0: 18118, 1: 556, 2: 1448, 3: 162, 4: 1608}


In [9]:
# ---- Tabel Check 1 (channels / sequence length / classes / label mapping)
def _range(lst):
    return f"{min(lst)}..{max(lst)}" if lst == list(range(min(lst), max(lst) + 1)) else str(lst)


check1 = pd.DataFrame([
    {"Check": "channels", "ECGTransForm": f"input_channels={CFG.input_channels} (Conv1d in_channels={_probe.conv1.in_channels})",
     "Dataset saya": "1  (setelah reshape (N,187) -> (N,1,187))", "Status": "PASS" if CFG.input_channels == 1 else "FAIL"},
    {"Check": "sequence length", "ECGTransForm": f"sequence_len={CFG.sequence_len} (tidak dipakai kode); model menerima L={_range(ok_L)}; 186->{T186['final_len']}, 187->{T187['final_len']}",
     "Dataset saya": f"{X_test.shape[1]}", "Status": "PASS" if (SEQ_COMPATIBLE and X_test.shape[1] == SEQ_LEN and X_test.shape[1] in ok_L) else "FAIL"},
    {"Check": "classes", "ECGTransForm": f"num_classes={CFG.num_classes}",
     "Dataset saya": f"{len(classes_test)} (test) / {len(classes_train)} (train)", "Status": "PASS" if CFG.num_classes == len(classes_test) == len(classes_train) == N_CLASSES else "FAIL"},
    {"Check": "label mapping", "ECGTransForm": f"{list(CFG.class_names)} (id = indeks)",
     "Dataset saya": f"{CLASS_NAMES} (0..4)", "Status": "PASS" if LABEL_OK else "FAIL"},
])
display(check1)

DATASET_OK = (X_test.shape == (N_TEST, SEQ_LEN) and y_test.shape == (N_TEST,) and X_test.dtype == np.float32
              and X_raw.shape == (N_TRAIN_RAW, SEQ_LEN) and RAW_SLICE_MATCHES_TABLE1 and len(classes_train) == N_CLASSES)
record(1, "Dataset", "PASS" if DATASET_OK else "FAIL",
       f"test {X_test.shape} {X_test.dtype}; train-mentah = X_train_smote[:87554] {X_raw.shape} (INFERENCE, distribusi = Table 1); "
       f"X_train_smote {X_train_smote.shape} = hasil SMOTE; raw CSV {RAW_CSV_STATUS}",
       "Train mentah diambil sebagai slice [:87554] (INFERENCE); data SMOTE tidak dianggap raw")
record(3, "Sequence length", "PASS" if check1.loc[1, "Status"] == "PASS" else "FAIL",
       f"sequence_len=186 tidak dipakai kode; L=186 & 187 -> panjang akhir {T187['final_len']} == trans_dim {CFG.trans_dim}; "
       f"L kompatibel {_range(ok_L)}",
       "Tidak ada (config tidak perlu diubah; input 187 titik utuh, tanpa dipotong)")

,Check,ECGTransForm,Dataset saya,Status
0,channels,input_channels=1 (Conv1d in_channels=1),"1 (setelah reshape (N,187) -> (N,1,187))",PASS
1,sequence length,"sequence_len=186 (tidak dipakai kode); model menerima L=180..187; 186->25, 187->25",187,PASS
2,classes,num_classes=5,5 (test) / 5 (train),PASS
3,label mapping,"['N', 'S', 'V', 'F', 'Q'] (id = indeks)","['N', 'S', 'V', 'F', 'Q'] (0..4)",PASS


[PASS] #1 Dataset: test (21892, 187) float32; train-mentah = X_train_smote[:87554] (87554, 187) (INFERENCE, distribusi = Table 1); X_train_smote (152471, 187) = hasil SMOTE; raw CSV tidak tersedia
[PASS] #3 Sequence length: sequence_len=186 tidak dipakai kode; L=186 & 187 -> panjang akhir 25 == trans_dim 25; L kompatibel 180..187


## Check 3 — Range / normalisasi

In [10]:
def stats(name, X):
    return {"set": name, "n": len(X), "min": float(X.min()), "max": float(X.max()),
            "mean": float(X.mean(dtype=np.float64)), "std": float(X.std(dtype=np.float64)),
            "NaN": int(np.isnan(X).sum()), "Inf": int(np.isinf(X).sum()),
            "kolom-187 == 0": float((X[:, -1] == 0).mean())}


stats_df = pd.DataFrame([
    stats("test (seluruh, 21.892)", X_test),
    stats("train mentah [:87554]", X_raw),
    stats("train sintetik (SMOTE)", X_syn),
    stats("X_train_smote (seluruh)", X_train_smote),
])
display(stats_df.round(4))

IN_UNIT_RANGE = bool((stats_df["min"] >= 0).all() and (stats_df["max"] <= 1).all())
ALL_FINITE = bool((stats_df["NaN"] == 0).all() and (stats_df["Inf"] == 0).all())
print("Semua set berada di [0, 1]?", IN_UNIT_RANGE, "| semua finite?", ALL_FINITE)
record(4, "Normalization", "PASS" if (IN_UNIT_RANGE and ALL_FINITE) else "FAIL",
       f"repo tidak menormalisasi (scan [3]); paper menormalisasi ke [0,1] saat preprocessing; dataset terukur "
       f"min={stats_df['min'].min():.3f} max={stats_df['max'].max():.3f}, NaN=0, Inf=0 pada test, train mentah, dan sintetik",
       "Tidak ada: normalisasi tambahan TIDAK diperlukan dan tidak diterapkan")

,set,n,min,max,mean,std,NaN,Inf,kolom-187 == 0
0,"test (seluruh, 21.892)",21892,0.0,1.0,0.1735,0.2256,0,0,0.9914
1,train mentah [:87554],87554,0.0,1.0,0.1743,0.2263,0,0,0.9910
2,train sintetik (SMOTE),64917,0.0,1.0,0.1815,0.2269,0,0,0.9362
3,X_train_smote (seluruh),152471,0.0,1.0,0.1774,0.2266,0,0,0.9677


Semua set berada di [0, 1]? True | semua finite? True
[PASS] #4 Normalization: repo tidak menormalisasi (scan [3]); paper menormalisasi ke [0,1] saat preprocessing; dataset terukur min=0.000 max=1.000, NaN=0, Inf=0 pada test, train mentah, dan sintetik


**Jawaban pertanyaan Check 3**

1. *Apakah ECGTransForm mengharapkan data sudah dinormalisasi?* — **INFERENCE**: ya. Model dan dataloader tidak punya normalisasi, sedangkan paper §4.1.3 menormalisasi amplitudo ke [0, 1] pada tahap preprocessing.
2. *Apakah repository melakukan normalisasi?* — **FACT**: tidak (scan [3]; `transforms` hanya di-import).
3. *Apakah dataset Kaggle sudah pada skala yang kompatibel?* — **FACT (terukur)**: ya, semua set berada di [0, 1] dan finite.
4. *Apakah normalisasi diperlukan?* — Tidak. Tidak ada transformasi yang ditambahkan.
5. *Apakah normalisasi membuat perbandingan dengan GAF tidak adil?* — Tidak ada normalisasi baru, jadi tidak ada perubahan definisi data. ECGTransForm membaca sinyal 1D yang sama dengan sumber GAF (`X_*.npy`); perbedaannya hanya modalitas input (1D vs citra GAF), yang memang inheren pada perbandingan metode. Test set tidak diubah.

# C. Check 4 — Preprocessing paper vs dataset Kaggle proyek

Tidak ada preprocessing raw-ECG yang dijalankan. Tabel ini hanya menentukan apakah tiap langkah **sudah tercermin**, **harus diulang**, atau **tidak boleh diulang**.

In [11]:
print("""
PAPER (ECGTransForm §4.1.3, mengikuti Kachuee dkk.)
  Raw MIT-BIH (360 Hz)
    -> window 10 s -> normalisasi amplitudo [0,1] -> kandidat R-peak (thr 0,9) -> median RR
    -> segmen 1,2*T per R-peak -> zero padding ke panjang tetap
    -> input model  ( .pt: {"samples","labels"} ; train / val / test )

PROJECT
  Kaggle MIT-BIH (shayanfazeli/heartbeat)
    -> dataset SUDAH berupa heartbeat 187 titik, nilai [0,1], ekor nol
    -> features/X_*.npy (+ SMOTE khusus train, Week 1)
    -> ECGTransForm adapter (notebook-only): reshape (N,187) -> (N,1,187), label -> int64
    -> model ECGTransForm (tidak dimodifikasi)
""")

pre = pd.DataFrame([
    {"Langkah": "R-peak detection",
     "Paper / repo": "Paper §4.1.3 langkah 3-4 (thr 0,9). Repo: tidak ada kode. [FACT]",
     "Dataset Kaggle proyek": f"Sampel sudah 1 detak; jumlah & distribusi kelas = Table 1 paper (train-mentah cocok: {RAW_SLICE_MATCHES_TABLE1}, test cocok: {dist_test == EXPECTED_TEST}). [FACT terukur]",
     "Status": "SUDAH tercermin (INFERENCE)", "Tindakan": "JANGAN diulang"},
    {"Langkah": "Heartbeat segmentation",
     "Paper / repo": "Paper langkah 5-6 (median RR, segmen 1,2*T). Repo: tidak ada. [FACT]",
     "Dataset Kaggle proyek": "Sudah tersegmentasi 187 titik/detak. [FACT: shape]",
     "Status": "SUDAH tercermin (INFERENCE)", "Tindakan": "JANGAN diulang"},
    {"Langkah": "Zero padding",
     "Paper / repo": "Paper langkah 7: padding ke panjang tetap (nilai panjang tidak disebut -> UNKNOWN). Repo: tidak ada.",
     "Dataset Kaggle proyek": f"Kolom ke-187 = 0 pada {last_col_zero_test:.1%} sampel test -> ekor nol. [FACT terukur]",
     "Status": "SUDAH tercermin (INFERENCE)", "Tindakan": "JANGAN diulang / jangan dipotong"},
    {"Langkah": "Normalization",
     "Paper / repo": "Paper langkah 2: amplitudo -> [0,1]. Repo: tidak ada. [FACT]",
     "Dataset Kaggle proyek": f"min={stats_df['min'].min():.3f}, max={stats_df['max'].max():.3f}. [FACT terukur]",
     "Status": "SUDAH tercermin", "Tindakan": "JANGAN diulang"},
    {"Langkah": "Denoising",
     "Paper / repo": "Tidak disebut di paper; repo: tidak ada.",
     "Dataset Kaggle proyek": "Tidak dapat dipastikan dari sumber ECGTransForm.",
     "Status": "UNKNOWN", "Tindakan": "Tidak dilakukan (tidak ada dasar)"},
    {"Langkah": "Filtering",
     "Paper / repo": "Tidak disebut di paper; repo: tidak ada.",
     "Dataset Kaggle proyek": "Tidak dapat dipastikan dari sumber ECGTransForm.",
     "Status": "UNKNOWN", "Tindakan": "Tidak dilakukan (tidak ada dasar)"},
    {"Langkah": "Resampling",
     "Paper / repo": "Paper menyebut rekaman 360 Hz, tidak menyebut resampling; repo: tidak ada.",
     "Dataset Kaggle proyek": "Paper rujukan proyek (Ahmad dkk.) menyebut versi Kaggle ini 125 Hz. Apakah penulis ECGTransForm memakai data identik tidak dapat diverifikasi.",
     "Status": "UNKNOWN", "Tindakan": "Tidak dilakukan"},
    {"Langkah": "Class balancing",
     "Paper / repo": "Paper §3.5.2: Context-Aware Loss (bobot kelas), bukan oversampling; SMOTE hanya pembanding ablasi (§5.4). Repo: tidak ada SMOTE. [FACT]",
     "Dataset Kaggle proyek": f"X_train_smote = SMOTE ({len(X_train_smote):,}); train mentah = 87.554 baris pertama (INFERENCE).",
     "Status": "BERBEDA dari protokol paper", "Tindakan": "Keputusan protokol (lihat Adaptasi #5)"},
])
display(pre)

UNKNOWNS.append("Denoising / filtering / resampling pada data penulis ECGTransForm tidak disebut di paper maupun repo -> kesetaraan dengan dataset Kaggle tidak dapat dipastikan.")
record(11, "Preprocessing paper vs Kaggle", "UNKNOWN",
       "Segmentasi, R-peak, zero padding, normalisasi [0,1] sudah tercermin (INFERENCE; Table 1 identik). "
       "Denoising / filtering / resampling: UNKNOWN. Tidak ada langkah yang perlu / boleh diulang.",
       "Tidak ada preprocessing tambahan")


PAPER (ECGTransForm §4.1.3, mengikuti Kachuee dkk.)
  Raw MIT-BIH (360 Hz)
    -> window 10 s -> normalisasi amplitudo [0,1] -> kandidat R-peak (thr 0,9) -> median RR
    -> segmen 1,2*T per R-peak -> zero padding ke panjang tetap
    -> input model  ( .pt: {"samples","labels"} ; train / val / test )

PROJECT
  Kaggle MIT-BIH (shayanfazeli/heartbeat)
    -> dataset SUDAH berupa heartbeat 187 titik, nilai [0,1], ekor nol
    -> features/X_*.npy (+ SMOTE khusus train, Week 1)
    -> ECGTransForm adapter (notebook-only): reshape (N,187) -> (N,1,187), label -> int64
    -> model ECGTransForm (tidak dimodifikasi)



,Langkah,Paper / repo,Dataset Kaggle proyek,Status,Tindakan
0,R-peak detection,"Paper §4.1.3 langkah 3-4 (thr 0,9). Repo: tidak ada kode. [FACT]","Sampel sudah 1 detak; jumlah & distribusi kelas = Table 1 paper (train-mentah cocok: True, test cocok: True). [FACT terukur]",SUDAH tercermin (INFERENCE),JANGAN diulang
1,Heartbeat segmentation,"Paper langkah 5-6 (median RR, segmen 1,2*T). Repo: tidak ada. [FACT]",Sudah tersegmentasi 187 titik/detak. [FACT: shape],SUDAH tercermin (INFERENCE),JANGAN diulang
2,Zero padding,Paper langkah 7: padding ke panjang tetap (nilai panjang tidak disebut -> UNKNOWN). Repo: tidak ada.,Kolom ke-187 = 0 pada 99.1% sampel test -> ekor nol. [FACT terukur],SUDAH tercermin (INFERENCE),JANGAN diulang / jangan dipotong
3,Normalization,"Paper langkah 2: amplitudo -> [0,1]. Repo: tidak ada. [FACT]","min=0.000, max=1.000. [FACT terukur]",SUDAH tercermin,JANGAN diulang
4,Denoising,Tidak disebut di paper; repo: tidak ada.,Tidak dapat dipastikan dari sumber ECGTransForm.,UNKNOWN,Tidak dilakukan (tidak ada dasar)
5,Filtering,Tidak disebut di paper; repo: tidak ada.,Tidak dapat dipastikan dari sumber ECGTransForm.,UNKNOWN,Tidak dilakukan (tidak ada dasar)
6,Resampling,"Paper menyebut rekaman 360 Hz, tidak menyebut resampling; repo: tidak ada.",Paper rujukan proyek (Ahmad dkk.) menyebut versi Kaggle ini 125 Hz. Apakah penulis ECGTransForm memakai data identik tidak dapat diverifikasi.,UNKNOWN,Tidak dilakukan
7,Class balancing,"Paper §3.5.2: Context-Aware Loss (bobot kelas), bukan oversampling; SMOTE hanya pembanding ablasi (§5.4). Repo: tidak ada SMOTE. [FACT]","X_train_smote = SMOTE (152,471); train mentah = 87.554 baris pertama (INFERENCE).",BERBEDA dari protokol paper,Keputusan protokol (lihat Adaptasi #5)


[UNKNOWN] #11 Preprocessing paper vs Kaggle: Segmentasi, R-peak, zero padding, normalisasi [0,1] sudah tercermin (INFERENCE; Table 1 identik). Denoising / filtering / resampling: UNKNOWN. Tidak ada langkah yang perlu / boleh diulang.


# D. Check 5 — Dataloader

Format yang diharapkan repo: dict `{"samples": (N,1,L) float, "labels": (N,) int}` di `train.pt` / `val.pt` / `test.pt`. Adapter di bawah **hanya** mengubah bentuk (`unsqueeze`) dan dtype label; nilai sinyal tidak disentuh. Kelas `Load_Dataset` dan fungsi `data_generator` yang dipakai adalah **kode asli repo**.

Untuk membuktikan format `.pt` benar-benar diterima `data_generator`, file `.pt` ditulis ke **folder sementara** (dihapus otomatis), dengan test set utuh dan validasi dipotong dari train-mentah. Split validasi ini hanya untuk demonstrasi format; tidak dipakai untuk hasil apa pun.

In [12]:
def to_ecgtransform_dict(X, y):
    """Adapter MINIMAL (notebook-only): (N,187) -> samples (N,1,187) float32, labels (N,) int64. Nilai TIDAK diubah."""
    return {
        "samples": torch.from_numpy(np.array(X, dtype=np.float32, copy=True)).unsqueeze(1),
        "labels": torch.from_numpy(np.array(y).astype(np.int64)),
    }


# ---- (a) Dataset ASLI repo + adapter pada train-mentah
ds = Load_Dataset(to_ecgtransform_dict(X_raw, y_raw))
s0 = ds[0]
print("Load_Dataset[0]:", {k: (tuple(v.shape), str(v.dtype)) if torch.is_tensor(v) else (type(v).__name__, v) for k, v in s0.items()})

# ---- (b) Bukti bahwa adapter DIPERLUKAN: tanpa sumbu channel, batch (B,187) tidak dapat masuk Conv1d
bad_ds = Load_Dataset({"samples": torch.from_numpy(np.array(X_raw[:16])), "labels": torch.from_numpy(y_raw[:16].copy())})
bad_x = next(iter(DataLoader(bad_ds, batch_size=AUDIT_BATCH)))["samples"]
try:
    with torch.no_grad():
        _probe.eval()(bad_x)
    NO_ADAPTER_FAILS, no_adapter_msg = False, "tidak error"
except Exception as e:  # noqa: BLE001
    NO_ADAPTER_FAILS, no_adapter_msg = True, f"{type(e).__name__}: {str(e).splitlines()[0][:150]}"
print(f"\nTANPA adapter: batch {tuple(bad_x.shape)} -> {no_adapter_msg}")

# ---- (c) 1 DataLoader kecil (batch = 8, berisi kelima kelas) dari train-mentah
rng = np.random.RandomState(SEED)
first_idx = [int(np.where(y_raw == c)[0][0]) for c in range(N_CLASSES)]
extra_idx = rng.choice(np.where(y_raw == 0)[0], AUDIT_BATCH - N_CLASSES, replace=False).tolist()
audit_idx = first_idx + extra_idx
audit_loader = DataLoader(Subset(ds, audit_idx), batch_size=AUDIT_BATCH, shuffle=False, num_workers=0)
audit_batch = next(iter(audit_loader))
x_audit, y_audit = audit_batch["samples"], audit_batch["labels"]
batch_values_unchanged = bool(np.array_equal(x_audit.squeeze(1).numpy(), np.asarray(X_raw[audit_idx])))
print(f"\nAudit batch: x {tuple(x_audit.shape)} {x_audit.dtype} | y {tuple(y_audit.shape)} {y_audit.dtype} | label = {y_audit.tolist()}")
print("Nilai batch identik dengan sumber (tanpa perubahan)?", batch_values_unchanged)

# ---- (d) `data_generator` ASLI dengan file .pt sementara (train / val / test)
with tempfile.TemporaryDirectory() as tmp:
    d = os.path.join(tmp, "mit")
    os.makedirs(d)
    tr_idx, va_idx = train_test_split(np.arange(N_TRAIN_RAW), test_size=0.2, stratify=y_raw, random_state=SEED)
    torch.save(to_ecgtransform_dict(X_raw[tr_idx], y_raw[tr_idx]), os.path.join(d, "train.pt"))
    torch.save(to_ecgtransform_dict(X_raw[va_idx], y_raw[va_idx]), os.path.join(d, "val.pt"))
    torch.save(to_ecgtransform_dict(X_test, y_test), os.path.join(d, "test.pt"))   # test set UTUH, tidak diubah

    train_dl, val_dl, test_dl, cw_split = data_generator(tmp, "mit", {"batch_size": AUDIT_BATCH})
    tb = next(iter(test_dl))
    x_test_b, y_test_b = tb["samples"].clone(), tb["labels"].clone()
    DG = dict(
        n_train=len(train_dl.dataset), n_val=len(val_dl.dataset), n_test=len(test_dl.dataset),
        test_x_equal=bool(torch.equal(test_dl.dataset.x_data.squeeze(1), torch.from_numpy(np.array(X_test)))),
        test_y_equal=bool(torch.equal(test_dl.dataset.y_data, torch.from_numpy(y_test.astype(np.int64)))),
        test_batch_shape=tuple(x_test_b.shape), test_batch_dtype=str(x_test_b.dtype),
    )
print("\ndata_generator (kode asli) dengan .pt sementara:", DG)
print("Folder sementara sudah dihapus. Tidak ada .pt yang tersimpan.")

# ---- (e) Bobot kelas (Context-Aware Loss) memakai fungsi ASLI repo
def repo_weights(counts):
    w = get_class_weight({int(k): int(v) for k, v in sorted(counts.items())})
    return [float(w[i]) for i in range(N_CLASSES)]


W_RAW, W_SMOTE = repo_weights(dist_raw), repo_weights(dist_smote)
display(pd.DataFrame({
    "kelas": CLASS_NAMES,
    "n train-mentah": [dist_raw[i] for i in range(N_CLASSES)], "bobot CAL (train-mentah)": np.round(W_RAW, 3),
    "n train-SMOTE": [dist_smote[i] for i in range(N_CLASSES)], "bobot CAL (train-SMOTE)": np.round(W_SMOTE, 3),
    "bobot baseline ResNet18": BASELINE_CLASS_WEIGHTS,
}))
print("FACT: bobot CAL bergantung pada distribusi train yang dipakai -> pilihan train-mentah vs train-SMOTE mengubah loss ECGTransForm.")

DATALOADER_OK = bool(
    tuple(x_audit.shape) == (AUDIT_BATCH, 1, SEQ_LEN) and x_audit.dtype == torch.float32 and y_audit.dtype == torch.int64
    and batch_values_unchanged and NO_ADAPTER_FAILS
    and DG["n_test"] == N_TEST and DG["test_x_equal"] and DG["test_y_equal"]
    and DG["test_batch_shape"] == (AUDIT_BATCH, 1, SEQ_LEN)
    and DG["n_train"] + DG["n_val"] == N_TRAIN_RAW
)
ADAPTATIONS += [
    dict(item="Reshape (N,187) -> (N,1,187)", level="minor",
         detail="Adapter notebook-only `to_ecgtransform_dict`; repo mensyaratkan sumbu channel. Nilai sinyal tidak berubah."),
    dict(item="Format .pt tidak wajib dibuat di Drive", level="minor",
         detail="`Load_Dataset` asli dapat memakai dict in-memory; untuk memakai `data_generator`, .pt ditulis ke folder sementara."),
    dict(item="Validation split dari train", level="protocol",
         detail="Repo mewajibkan val.pt (paper §4.2: 20% train). Proyek tidak punya validation set -> harus dipotong dari train (BUKAN dari test), seed tetap, stratified."),
    dict(item="Pilih train set: mentah 87.554 vs SMOTE 152.471", level="protocol",
         detail="Paper/repo tidak memakai SMOTE (bobot CAL, bukan oversampling) dan train-mentah hanya tersedia sebagai slice [:87554] (INFERENCE). "
                "Baseline memakai data SMOTE. Bobot CAL berbeda untuk kedua pilihan (tabel di atas)."),
    dict(item="Protokol seleksi checkpoint / evaluasi", level="protocol",
         detail="Repo: pilih epoch dengan F1 validasi, test dievaluasi di epoch terakhir. Week 1: checkpoint terbaik dipilih berdasarkan metrik test. "
                "Kedua protokol berbeda -> harus ditetapkan sebelum training agar perbandingan adil."),
]
record(5, "Dataloader", "PASS" if DATALOADER_OK else "FAIL",
       f"Load_Dataset & data_generator asli menerima dict {{samples (N,1,187), labels int64}}; batch {tuple(x_audit.shape)}; "
       f"test via data_generator = {DG['n_test']} sampel identik dengan X_test; tanpa adapter batch (8,187) gagal",
       "Adapter reshape + cast int64; val split dari train bila memakai data_generator (lihat tabel Adaptasi)")

Load_Dataset[0]: {'samples': ((1, 187), 'torch.float32'), 'labels': ('int', 0)}

TANPA adapter: batch (8, 187) -> RuntimeError: Given groups=1, weight of size [32, 1, 5], expected input[1, 8, 187] to have 1 channels, but got 8 channels instead

Audit batch: x (8, 1, 187) torch.float32 | y (8,) torch.int64 | label = [0, 1, 2, 3, 4, 0, 0, 0]
Nilai batch identik dengan sumber (tanpa perubahan)? True



data_generator (kode asli) dengan .pt sementara: {'n_train': 70043, 'n_val': 17511, 'n_test': 21892, 'test_x_equal': True, 'test_y_equal': True, 'test_batch_shape': (8, 1, 187), 'test_batch_dtype': 'torch.float32'}
Folder sementara sudah dihapus. Tidak ada .pt yang tersimpan.


,kelas,n train-mentah,bobot CAL (train-mentah),n train-SMOTE,bobot CAL (train-SMOTE),bobot baseline ResNet18
0,N,72471,1.000,72471,1.000,1
1,S,2223,3.484,30000,1.000,2
2,V,5788,2.527,20000,1.287,1
3,F,641,4.728,20000,1.287,2
4,Q,6431,2.422,10000,1.981,1


FACT: bobot CAL bergantung pada distribusi train yang dipakai -> pilihan train-mentah vs train-SMOTE mengubah loss ECGTransForm.
[PASS] #5 Dataloader: Load_Dataset & data_generator asli menerima dict {samples (N,1,187), labels int64}; batch (8, 1, 187); test via data_generator = 21892 sampel identik dengan X_test; tanpa adapter batch (8,187) gagal


# E. Check 6–9 — Model, forward, loss, backward

Semua memakai `ecgTransForm` asli dengan bobot **acak** (`seed=42`). Tidak ada training. Satu batch kecil, satu forward / loss / backward, **satu** `optimizer.step()`, lalu berhenti.

In [13]:
torch.manual_seed(SEED)
model = ecgTransForm(configs=CFG, hparams=HP).to(DEVICE)   # kode asli, tanpa modifikasi
n_params = sum(p.numel() for p in model.parameters())
n_unused_params = sum(p.numel() for n, p in model.named_parameters() if n.startswith("encoder_layer."))

arch = pd.DataFrame([
    {"Komponen": "Input channel", "Nilai": f"conv1/2/3 in_channels = {model.conv1.in_channels}"},
    {"Komponen": "Konvolusi multi-skala", "Nilai": f"kernel {[model.conv1.kernel_size[0], model.conv2.kernel_size[0], model.conv3.kernel_size[0]]}, {CFG.mid_channels} filter, padding 'same' (stride {CFG.stride}) -> dirata-rata"},
    {"Komponen": "Blok conv 2 / 3", "Nilai": f"kernel 8: {model.conv_block2[0].in_channels}->{model.conv_block2[0].out_channels}, {model.conv_block3[0].in_channels}->{model.conv_block3[0].out_channels}"},
    {"Komponen": "Pooling", "Nilai": f"MaxPool1d kernel={model.mp.kernel_size}, stride={model.mp.stride}, padding={model.mp.padding} (3 kali total)"},
    {"Komponen": "Recalibration (attention kanal)", "Nilai": f"CRM = {len(model.crm)} x SE-residual block ({model.crm[0].se.fc[0].in_features} kanal, reduction 4)"},
    {"Komponen": "Transformer", "Nilai": f"d_model={model.encoder_layer.self_attn.embed_dim}, heads={model.encoder_layer.self_attn.num_heads}, layers={model.transformer_encoder.num_layers}, batch_first={model.encoder_layer.self_attn.batch_first}, dua arah (flip sumbu waktu)"},
    {"Komponen": "Dimensi fitur akhir", "Nilai": f"AdaptiveAvgPool1d(1) -> {model.clf.in_features}  (hparams feature_dim={HP['feature_dim']})"},
    {"Komponen": "Output classifier", "Nilai": f"Linear({model.clf.in_features}, {model.clf.out_features})"},
    {"Komponen": "Jumlah parameter", "Nilai": f"{n_params:,} (termasuk {n_unused_params:,} parameter `encoder_layer` yang tidak dipakai langsung)"},
])
display(arch)

# ---- Trace shape per tahap untuk input (8, 1, 187)
T = trace(model, SEQ_LEN, batch=AUDIT_BATCH)
stage_df = pd.DataFrame(
    [{"tahap": "input", "shape": (AUDIT_BATCH, CFG.input_channels, SEQ_LEN)}]
    + [{"tahap": k, "shape": v} for k, v in T["shapes"].items()]
    + [{"tahap": "output (logits)", "shape": T["out"]}]
)
display(stage_df)

MODEL_INPUT_OK = bool(
    T["ok"] and T["out"] == (AUDIT_BATCH, N_CLASSES)
    and T["shapes"]["conv_block3"] == (AUDIT_BATCH, CFG.final_out_channels, CFG.trans_dim)
    and model.clf.in_features == CFG.final_out_channels == HP["feature_dim"]
)
print("Model menerima (batch,1,187) dan menghasilkan (batch,5)?", MODEL_INPUT_OK)
print("""
INFERENCE : Transformer bekerja pada tensor (B, 128, 25) dengan batch_first=True -> 128 "token" kanal, masing-masing
            berdimensi 25 (sumbu waktu). Karena itu panjang waktu akhir harus tepat 25 (= d_model).
Modifikasi: TIDAK ADA pada models.py / data_configs.py.""")
record(6, "Model input", "PASS" if MODEL_INPUT_OK else "FAIL",
       f"(8,1,187) -> conv_block3 {T['shapes'].get('conv_block3')} (panjang waktu {CFG.trans_dim} = d_model) -> logits {T['out']}",
       "Tidak ada perubahan pada model; config.sequence_len=186 dibiarkan (tidak dibaca kode)")

/home/claude/work/wd/ECGTransForm/models.py:41: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=3)


,Komponen,Nilai
0,Input channel,conv1/2/3 in_channels = 1
1,Konvolusi multi-skala,"kernel [5, 9, 11], 32 filter, padding 'same' (stride 1) -> dirata-rata"
2,Blok conv 2 / 3,"kernel 8: 32->64, 64->128"
3,Pooling,"MaxPool1d kernel=2, stride=2, padding=1 (3 kali total)"
4,Recalibration (attention kanal),"CRM = 3 x SE-residual block (128 kanal, reduction 4)"
5,Transformer,"d_model=25, heads=5, layers=3, batch_first=True, dua arah (flip sumbu waktu)"
6,Dimensi fitur akhir,AdaptiveAvgPool1d(1) -> 128 (hparams feature_dim=128)
7,Output classifier,"Linear(128, 5)"
8,Jumlah parameter,"637,689 (termasuk 107,173 parameter `encoder_layer` yang tidak dipakai langsung)"


,tahap,shape
0,input,"(8, 1, 187)"
1,mp,"(8, 32, 94)"
2,conv_block2,"(8, 64, 48)"
3,conv_block3,"(8, 128, 25)"
4,crm,"(8, 128, 25)"
5,transformer_encoder,"(8, 128, 25)"
6,aap,"(8, 128, 1)"
7,clf,"(8, 5)"
8,output (logits),"(8, 5)"


Model menerima (batch,1,187) dan menghasilkan (batch,5)? True

INFERENCE : Transformer bekerja pada tensor (B, 128, 25) dengan batch_first=True -> 128 "token" kanal, masing-masing
            berdimensi 25 (sumbu waktu). Karena itu panjang waktu akhir harus tepat 25 (= d_model).
Modifikasi: TIDAK ADA pada models.py / data_configs.py.
[PASS] #6 Model input: (8,1,187) -> conv_block3 (8, 128, 25) (panjang waktu 25 = d_model) -> logits (8, 5)


In [14]:
# ---- Check 7: FORWARD PASS (eval, no_grad) pada batch kecil
def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def forward_eval(xb):
    sync()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model(xb)
    sync()
    return out, (time.perf_counter() - t0) * 1000


model.eval()
fw_rows = []
for name, xb in (("batch train-mentah (5 kelas)", x_audit), ("batch test (8 sampel pertama, forward saja)", x_test_b)):
    xb = xb.to(DEVICE)
    forward_eval(xb)                                   # warm-up
    times = sorted(forward_eval(xb)[1] for _ in range(10))
    out, _ = forward_eval(xb)
    fw_rows.append({"batch": name, "input shape": tuple(xb.shape), "input dtype": str(xb.dtype),
                    "output shape": tuple(out.shape), "output dtype": str(out.dtype),
                    "NaN": bool(torch.isnan(out).any()), "Inf": bool(torch.isinf(out).any()),
                    "waktu forward (ms, median 10x)": round(times[len(times) // 2], 2)})
display(pd.DataFrame(fw_rows))
FORWARD_OK = all(r["output shape"] == (AUDIT_BATCH, N_CLASSES) and not r["NaN"] and not r["Inf"] for r in fw_rows)
record(7, "Forward pass", "PASS" if FORWARD_OK else "FAIL",
       f"input (8,1,187) -> output (8,5) float32, tanpa NaN/Inf; waktu {fw_rows[0]['waktu forward (ms, median 10x)']} ms ({DEVICE.type}, batch 8, eval)",
       "Tidak ada")

,batch,input shape,input dtype,output shape,output dtype,NaN,Inf,"waktu forward (ms, median 10x)"
0,batch train-mentah (5 kelas),"(8, 1, 187)",torch.float32,"(8, 5)",torch.float32,False,False,33.80
1,"batch test (8 sampel pertama, forward saja)","(8, 1, 187)",torch.float32,"(8, 5)",torch.float32,False,False,34.03


[PASS] #7 Forward pass: input (8,1,187) -> output (8,5) float32, tanpa NaN/Inf; waktu 33.8 ms (cpu, batch 8, eval)


In [15]:
# ---- Check 8: LOSS (CrossEntropyLoss berbobot CAL, seperti trainer.py asli), mode train
criterion = nn.CrossEntropyLoss(weight=torch.tensor(W_RAW, dtype=torch.float32, device=DEVICE))
model.train()
x, y = x_audit.to(DEVICE), y_audit.to(DEVICE)
logits = model(x)
loss = criterion(logits, y)

print(f"logits shape : {tuple(logits.shape)} {logits.dtype}")
print(f"label shape  : {tuple(y.shape)} {y.dtype}  (nilai {y.tolist()})")
print(f"loss = {loss.item():.6f}   finite = {bool(torch.isfinite(loss))}")
LOSS_OK = bool(tuple(logits.shape) == (AUDIT_BATCH, N_CLASSES) and tuple(y.shape) == (AUDIT_BATCH,)
               and y.dtype == torch.int64 and torch.isfinite(loss))
record(8, "Loss", "PASS" if LOSS_OK else "FAIL",
       f"CrossEntropyLoss(weight=CAL train-mentah): logits {tuple(logits.shape)}, label {tuple(y.shape)} int64, loss={loss.item():.4f} (finite)",
       "Tidak ada")

logits shape : (8, 5) torch.float32
label shape  : (8,) torch.int64  (nilai [0, 1, 2, 3, 4, 0, 0, 0])
loss = 1.614596   finite = True
[PASS] #8 Loss: CrossEntropyLoss(weight=CAL train-mentah): logits (8, 5), label (8,) int64, loss=1.6146 (finite)


In [16]:
# ---- Check 9: BACKWARD + satu optimizer.step()  (lalu STOP)
optimizer = torch.optim.Adam(model.parameters(), lr=HP["learning_rate"], weight_decay=HP["weight_decay"], betas=(0.9, 0.99))  # sama dengan trainer.py
optimizer.zero_grad()
loss.backward()

named = list(model.named_parameters())
none_grad = [n for n, p in named if p.grad is None]
unexpected_none = [n for n in none_grad if not n.startswith("encoder_layer.")]
nonfinite_grad = [n for n, p in named if p.grad is not None and not torch.isfinite(p.grad).all()]
grad_norm = torch.sqrt(sum((p.grad.float() ** 2).sum() for _, p in named if p.grad is not None)).item()

before = {n: p.detach().clone() for n, p in named}
optimizer.step()
n_changed = sum(not torch.equal(before[n], p.detach()) for n, p in named)
finite_after = all(bool(torch.isfinite(p).all()) for _, p in named)

print(f"Parameter total / punya gradien / tanpa gradien : {len(named)} / {len(named) - len(none_grad)} / {len(none_grad)}")
print(f"  tanpa gradien di luar `encoder_layer.*`        : {unexpected_none or 'tidak ada'}")
print(f"  (semua {len(none_grad)} param tanpa gradien berawalan `encoder_layer.`: {not unexpected_none} -> salinan tak terpakai, jinak)")
print(f"Gradien NaN/Inf                                  : {nonfinite_grad or 'tidak ada'}")
print(f"Norma gradien global                             : {grad_norm:.4f}")
print(f"Parameter yang berubah setelah 1x optimizer.step : {n_changed} / {len(named) - len(none_grad)}")
print(f"Semua parameter finite setelah step              : {finite_after}")
print("\nSTOP. Tidak ada step lanjutan, tidak ada epoch, tidak ada checkpoint yang disimpan.")

BACKWARD_OK = bool(not unexpected_none and not nonfinite_grad and n_changed == len(named) - len(none_grad) and finite_after)
record(9, "Backward", "PASS" if BACKWARD_OK else "FAIL",
       f"loss.backward() tanpa error; gradien finite (norma {grad_norm:.3f}); {len(named) - len(none_grad)}/{len(named)} parameter menerima gradien "
       f"(sisanya = encoder_layer.* salinan tak terpakai); 1x Adam.step mengubah {n_changed} parameter, semuanya finite",
       "Tidak ada")

Parameter total / punya gradien / tanpa gradien : 91 / 79 / 12
  tanpa gradien di luar `encoder_layer.*`        : tidak ada
  (semua 12 param tanpa gradien berawalan `encoder_layer.`: True -> salinan tak terpakai, jinak)
Gradien NaN/Inf                                  : tidak ada
Norma gradien global                             : 0.1329
Parameter yang berubah setelah 1x optimizer.step : 79 / 79
Semua parameter finite setelah step              : True

STOP. Tidak ada step lanjutan, tidak ada epoch, tidak ada checkpoint yang disimpan.
[PASS] #9 Backward: loss.backward() tanpa error; gradien finite (norma 0.133); 79/91 parameter menerima gradien (sisanya = encoder_layer.* salinan tak terpakai); 1x Adam.step mengubah 79 parameter, semuanya finite


# F. Check 10 & 11 — Integritas test set dan repository

In [17]:
# ---- Check 10: TEST SET INTEGRITY (dibaca ulang dari disk)
X_test_disk, y_test_disk = np.load(PATHS["X_test.npy"]), np.load(PATHS["y_test.npy"])
integrity = {
    "sha256 file X_test.npy sama": sha256_file(PATHS["X_test.npy"]) == TEST_BEFORE["x_file_sha256"],
    "sha256 file y_test.npy sama": sha256_file(PATHS["y_test.npy"]) == TEST_BEFORE["y_file_sha256"],
    "sha256 array X_test (memori) sama": sha256_array(X_test) == TEST_BEFORE["x_array_sha256"],
    "sha256 array y_test (memori) sama": sha256_array(y_test) == TEST_BEFORE["y_array_sha256"],
    "shape X sama": X_test_disk.shape == TEST_BEFORE["x_shape"] == (N_TEST, SEQ_LEN),
    "shape y sama": y_test_disk.shape == TEST_BEFORE["y_shape"] == (N_TEST,),
    "jumlah sampel = 21.892": len(X_test_disk) == len(y_test_disk) == N_TEST,
    "distribusi label sama & = target": dist(y_test_disk) == TEST_BEFORE["dist"] == EXPECTED_TEST,
    "nilai identik dengan snapshot di memori": bool(np.array_equal(X_test_disk, X_test) and np.array_equal(y_test_disk, y_test)),
    "array audit read-only": (not X_test.flags.writeable) and (not y_test.flags.writeable),
}
display(pd.DataFrame({"pemeriksaan": list(integrity), "hasil": ["PASS" if v else "FAIL" for v in integrity.values()]}))
print("Distribusi test setelah audit:", {CLASS_NAMES[k]: v for k, v in dist(y_test_disk).items()})
TEST_UNCHANGED = all(integrity.values())
record(10, "Test integrity", "PASS" if TEST_UNCHANGED else "FAIL",
       f"21.892 x 187; N/S/V/F/Q = {[dist(y_test_disk)[i] for i in range(5)]}; hash file & array identik sebelum/sesudah audit",
       "Tidak ada (test set tidak di-resample / di-SMOTE / dipotong / dibuat ulang)")

,pemeriksaan,hasil
0,sha256 file X_test.npy sama,PASS
1,sha256 file y_test.npy sama,PASS
2,sha256 array X_test (memori) sama,PASS
3,sha256 array y_test (memori) sama,PASS
4,shape X sama,PASS
5,shape y sama,PASS
6,jumlah sampel = 21.892,PASS
7,distribusi label sama & = target,PASS
8,nilai identik dengan snapshot di memori,PASS
9,array audit read-only,PASS


Distribusi test setelah audit: {'N': 18118, 'S': 556, 'V': 1448, 'F': 162, 'Q': 1608}
[PASS] #10 Test integrity: 21.892 x 187; N/S/V/F/Q = [18118, 556, 1448, 162, 1608]; hash file & array identik sebelum/sesudah audit


In [18]:
# ---- Check 11: REPOSITORY INTEGRITY
REPO_AFTER = repo_snapshot()
changed = sorted(f for f in REPO_BEFORE if f in REPO_AFTER and REPO_BEFORE[f] != REPO_AFTER[f])
added = sorted(set(REPO_AFTER) - set(REPO_BEFORE))
removed = sorted(set(REPO_BEFORE) - set(REPO_AFTER))
git_status = subprocess.run(["git", "-C", REPO_DIR, "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip()
HEAD_AFTER = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()

print("File repo berubah :", changed or "tidak ada")
print("File repo ditambah:", added or "tidak ada")
print("File repo dihapus :", removed or "tidak ada")
print("git status        :", git_status or "bersih")
print("HEAD              :", HEAD_AFTER == REPO_COMMIT)
REPO_CLEAN = not (changed or added or removed or git_status) and HEAD_AFTER == REPO_COMMIT

print("\nFile yang diadaptasi / dibuat oleh audit ini:")
print("  - TIDAK ADA file repo yang diedit. Adapter = fungsi `to_ecgtransform_dict` di notebook ini (notebook-only wrapper).")
print("  - Folder `ecgtransform_adapter/` TIDAK dibuat (tidak diperlukan).")
print("  - .pt sementara ditulis ke folder temp dan dihapus otomatis.")
record(12, "Repository integrity", "PASS" if REPO_CLEAN else "FAIL",
       f"commit {REPO_COMMIT[:7]}; hash {len(REPO_BEFORE)} file sama sebelum/sesudah; git status bersih",
       "Tidak ada file repo diubah; adapter hanya di notebook")

File repo berubah : tidak ada
File repo ditambah: tidak ada
File repo dihapus : tidak ada
git status        : bersih
HEAD              : True

File yang diadaptasi / dibuat oleh audit ini:
  - TIDAK ADA file repo yang diedit. Adapter = fungsi `to_ecgtransform_dict` di notebook ini (notebook-only wrapper).
  - Folder `ecgtransform_adapter/` TIDAK dibuat (tidak diperlukan).
  - .pt sementara ditulis ke folder temp dan dihapus otomatis.
[PASS] #12 Repository integrity: commit 2f5c28c; hash 12 file sama sebelum/sesudah; git status bersih


# G. Hasil akhir

In [19]:
final = pd.DataFrame([{"No": n, "Audit": a["audit"], "Status": a["status"], "Evidence": a["evidence"], "Adaptasi": a["adaptation"]}
                      for n, a in sorted(AUDIT.items())])
display(final.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

CRITICAL = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12]
failed = [n for n in CRITICAL if AUDIT[n]["status"] != "PASS"]
protocol_items = [a for a in ADAPTATIONS if a["level"] == "protocol"]

if failed:
    VERDICT = "NOT FEASIBLE"
    RECOMMENDATION = "GANTI METODE" if {3, 6} & set(failed) else "PERBAIKI ADAPTER TERLEBIH DAHULU"
elif protocol_items:
    VERDICT = "FEASIBLE WITH ADAPTATION"
    RECOMMENDATION = "LANJUT TRAINING (bersyarat: tetapkan dan dokumentasikan keputusan protokol di bawah SEBELUM training)"
else:
    VERDICT = "FEASIBLE"
    RECOMMENDATION = "LANJUT TRAINING"

print(f"Pemeriksaan kritis yang tidak PASS: {failed or 'tidak ada'}")
print(f"\n### KEPUTUSAN AKHIR: {VERDICT}")
print(f"### REKOMENDASI    : {RECOMMENDATION}")
print("\nCatatan: keputusan ini menilai KELAYAKAN TEKNIS saja. Audit tidak menguji performa, "
      "tidak menyimpulkan metode mana yang lebih baik, dan tidak mengklaim reproduksi paper.")

No,Audit,Status,Evidence,Adaptasi
1,Dataset,PASS,"test (21892, 187) float32; train-mentah = X_train_smote[:87554] (87554, 187) (INFERENCE, distribusi = Table 1); X_train_smote (152471, 187) = hasil SMOTE; raw CSV tidak tersedia",Train mentah diambil sebagai slice [:87554] (INFERENCE); data SMOTE tidak dianggap raw
2,Label,PASS,"repo class_names=['N', 'S', 'V', 'F', 'Q'] == proyek; num_classes=5; test dist == target {0: 18118, 1: 556, 2: 1448, 3: 162, 4: 1608}",Tidak ada remapping; hanya cast dtype ke int64
3,Sequence length,PASS,sequence_len=186 tidak dipakai kode; L=186 & 187 -> panjang akhir 25 == trans_dim 25; L kompatibel 180..187,"Tidak ada (config tidak perlu diubah; input 187 titik utuh, tanpa dipotong)"
4,Normalization,PASS,"repo tidak menormalisasi (scan [3]); paper menormalisasi ke [0,1] saat preprocessing; dataset terukur min=0.000 max=1.000, NaN=0, Inf=0 pada test, train mentah, dan sintetik",Tidak ada: normalisasi tambahan TIDAK diperlukan dan tidak diterapkan
5,Dataloader,PASS,"Load_Dataset & data_generator asli menerima dict {samples (N,1,187), labels int64}; batch (8, 1, 187); test via data_generator = 21892 sampel identik dengan X_test; tanpa adapter batch (8,187) gagal",Adapter reshape + cast int64; val split dari train bila memakai data_generator (lihat tabel Adaptasi)
6,Model input,PASS,"(8,1,187) -> conv_block3 (8, 128, 25) (panjang waktu 25 = d_model) -> logits (8, 5)",Tidak ada perubahan pada model; config.sequence_len=186 dibiarkan (tidak dibaca kode)
7,Forward pass,PASS,"input (8,1,187) -> output (8,5) float32, tanpa NaN/Inf; waktu 33.8 ms (cpu, batch 8, eval)",Tidak ada
8,Loss,PASS,"CrossEntropyLoss(weight=CAL train-mentah): logits (8, 5), label (8,) int64, loss=1.6146 (finite)",Tidak ada
9,Backward,PASS,"loss.backward() tanpa error; gradien finite (norma 0.133); 79/91 parameter menerima gradien (sisanya = encoder_layer.* salinan tak terpakai); 1x Adam.step mengubah 79 parameter, semuanya finite",Tidak ada
10,Test integrity,PASS,"21.892 x 187; N/S/V/F/Q = [18118, 556, 1448, 162, 1608]; hash file & array identik sebelum/sesudah audit",Tidak ada (test set tidak di-resample / di-SMOTE / dipotong / dibuat ulang)


Pemeriksaan kritis yang tidak PASS: tidak ada

### KEPUTUSAN AKHIR: FEASIBLE WITH ADAPTATION
### REKOMENDASI    : LANJUT TRAINING (bersyarat: tetapkan dan dokumentasikan keputusan protokol di bawah SEBELUM training)

Catatan: keputusan ini menilai KELAYAKAN TEKNIS saja. Audit tidak menguji performa, tidak menyimpulkan metode mana yang lebih baik, dan tidak mengklaim reproduksi paper.


In [20]:
print("=" * 100, "\n1. RINGKASAN TEMUAN SUMBER (rinci: tabel FACT/INFERENCE/UNKNOWN di Section A)\n", "=" * 100, sep="")
print(f"""
 data_configs.py : kelas 5 (N,S,V,F,Q), channel 1, trans_dim 25; `sequence_len={CFG.sequence_len}` TIDAK dipakai kode (FACT).
 dataloader.py   : membaca train/val/test .pt berisi {{samples (N,1,L), labels}}; wajib ada val.pt; tanpa normalisasi/transform;
                   bobot kelas CAL dari distribusi train (FACT).
 models.py       : CNN multi-skala + SE + Transformer dua arah; panjang waktu akhir harus 25 -> input {_range(ok_L)} titik diterima,
                   termasuk 186 dan 187 (diverifikasi numerik).
 paper           : preprocessing = Kachuee dkk. (window 10 s, normalisasi [0,1], R-peak, segmen 1,2T, zero padding) -> sudah tercermin
                   pada dataset Kaggle (INFERENCE; Table 1 identik dengan split proyek). Panjang tetap, denoising, filtering,
                   resampling tidak disebut (UNKNOWN).
""")

print("=" * 100, "\n2. TABEL PASS / FAIL / UNKNOWN: lihat tabel di atas\n", "=" * 100, sep="")

print("=" * 100, "\n3. PERUBAHAN YANG DIPERLUKAN DARI IMPLEMENTASI ASLI\n", "=" * 100, sep="")
print("Perubahan pada KODE ECGTransForm asli (models.py, dataloader.py, configs): TIDAK ADA.\n")
display(pd.DataFrame(ADAPTATIONS).rename(columns={"item": "Adaptasi", "level": "Tingkat", "detail": "Rincian"}).style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
print("Tingkat 'minor' = perubahan bentuk/dtype pada sisi data. Tingkat 'protocol' = keputusan desain eksperimen yang harus ditetapkan pengguna.")

print("\n" + "=" * 100, "\n4. Apakah dataset Kaggle 87.554/21.892 dapat dipakai untuk ECGTransForm tanpa mengubah test set?\n", "=" * 100, sep="")
q4 = TEST_UNCHANGED and DATASET_OK and DATALOADER_OK
print(f"{'YA' if q4 else 'TIDAK / BELUM TERBUKTI'}. Test set 21.892 x 187 dimasukkan utuh (data_generator memuatnya identik) dan hash file/array tidak berubah.")
print("Syarat/catatan: (a) train-mentah 87.554 hanya tersedia sebagai slice [:87554] dari data SMOTE (INFERENCE, bukan raw CSV);")
print("                (b) repo butuh val.pt -> dipotong dari TRAIN, bukan dari test; (c) pilihan train mentah vs SMOTE harus diputuskan.")

print("\n" + "=" * 100, "\n5. Apakah ECGTransForm dapat dijalankan dengan input 187 titik dan 5 kelas N/S/V/F/Q setelah adaptasi minimal?\n", "=" * 100, sep="")
q5 = MODEL_INPUT_OK and FORWARD_OK and LOSS_OK and BACKWARD_OK
print(f"{'YA' if q5 else 'TIDAK / BELUM TERBUKTI'}. Forward (8,1,187)->(8,5), loss finite ({loss.item():.4f}), backward + 1x optimizer.step berhasil; "
      "adaptasi pada sisi data hanya reshape + cast label; model tidak dimodifikasi.")
print("Ini hanya bukti kelayakan teknis pada bobot acak; BUKAN bukti performa.")

print("\n" + "=" * 100, "\n6. REKOMENDASI\n", "=" * 100, sep="")
print(RECOMMENDATION)
if not failed:
    print("Keputusan protokol yang perlu ditetapkan sebelum training:")
    for a in protocol_items:
        print(f"  - {a['item']}: {a['detail']}")

print("\nRisiko residual (UNKNOWN, tidak mengubah status kelayakan teknis):")
for u in UNKNOWNS:
    print("  -", u)

1. RINGKASAN TEMUAN SUMBER (rinci: tabel FACT/INFERENCE/UNKNOWN di Section A)

 data_configs.py : kelas 5 (N,S,V,F,Q), channel 1, trans_dim 25; `sequence_len=186` TIDAK dipakai kode (FACT).
 dataloader.py   : membaca train/val/test .pt berisi {samples (N,1,L), labels}; wajib ada val.pt; tanpa normalisasi/transform;
                   bobot kelas CAL dari distribusi train (FACT).
 models.py       : CNN multi-skala + SE + Transformer dua arah; panjang waktu akhir harus 25 -> input 180..187 titik diterima,
                   termasuk 186 dan 187 (diverifikasi numerik).
 paper           : preprocessing = Kachuee dkk. (window 10 s, normalisasi [0,1], R-peak, segmen 1,2T, zero padding) -> sudah tercermin
                   pada dataset Kaggle (INFERENCE; Table 1 identik dengan split proyek). Panjang tetap, denoising, filtering,
                   resampling tidak disebut (UNKNOWN).

2. TABEL PASS / FAIL / UNKNOWN: lihat tabel di atas
3. PERUBAHAN YANG DIPERLUKAN DARI IMPLEMENTASI ASLI
Peruba

Adaptasi,Tingkat,Rincian
Cast label ke int64,minor,y_train_smote float64 dan y_test int32 -> int64; nilai label tidak berubah (Load_Dataset juga memanggil .long()).
"Reshape (N,187) -> (N,1,187)",minor,Adapter notebook-only `to_ecgtransform_dict`; repo mensyaratkan sumbu channel. Nilai sinyal tidak berubah.
Format .pt tidak wajib dibuat di Drive,minor,"`Load_Dataset` asli dapat memakai dict in-memory; untuk memakai `data_generator`, .pt ditulis ke folder sementara."
Validation split dari train,protocol,"Repo mewajibkan val.pt (paper §4.2: 20% train). Proyek tidak punya validation set -> harus dipotong dari train (BUKAN dari test), seed tetap, stratified."
Pilih train set: mentah 87.554 vs SMOTE 152.471,protocol,"Paper/repo tidak memakai SMOTE (bobot CAL, bukan oversampling) dan train-mentah hanya tersedia sebagai slice [:87554] (INFERENCE). Baseline memakai data SMOTE. Bobot CAL berbeda untuk kedua pilihan (tabel di atas)."
Protokol seleksi checkpoint / evaluasi,protocol,"Repo: pilih epoch dengan F1 validasi, test dievaluasi di epoch terakhir. Week 1: checkpoint terbaik dipilih berdasarkan metrik test. Kedua protokol berbeda -> harus ditetapkan sebelum training agar perbandingan adil."


Tingkat 'minor' = perubahan bentuk/dtype pada sisi data. Tingkat 'protocol' = keputusan desain eksperimen yang harus ditetapkan pengguna.

4. Apakah dataset Kaggle 87.554/21.892 dapat dipakai untuk ECGTransForm tanpa mengubah test set?
YA. Test set 21.892 x 187 dimasukkan utuh (data_generator memuatnya identik) dan hash file/array tidak berubah.
Syarat/catatan: (a) train-mentah 87.554 hanya tersedia sebagai slice [:87554] dari data SMOTE (INFERENCE, bukan raw CSV);
                (b) repo butuh val.pt -> dipotong dari TRAIN, bukan dari test; (c) pilihan train mentah vs SMOTE harus diputuskan.

5. Apakah ECGTransForm dapat dijalankan dengan input 187 titik dan 5 kelas N/S/V/F/Q setelah adaptasi minimal?
YA. Forward (8,1,187)->(8,5), loss finite (1.6146), backward + 1x optimizer.step berhasil; adaptasi pada sisi data hanya reshape + cast label; model tidak dimodifikasi.
Ini hanya bukti kelayakan teknis pada bobot acak; BUKAN bukti performa.

6. REKOMENDASI
LANJUT TRAINING (bersyarat: te

In [21]:
# ---- Simpan hasil audit (opsional): satu file JSON kecil, file baru
def _jd(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


report = {
    "audit_date": datetime.now().isoformat(timespec="seconds"),
    "scope": "compatibility audit only; no training, no performance evaluation, no claim of paper reproduction",
    "repo": {"url": REPO_URL, "commit": REPO_COMMIT},
    "dataset": {
        "shapes": {"X_train_smote": X_train_smote.shape, "y_train_smote": y_train_smote.shape, "X_test": X_test.shape, "y_test": y_test.shape,
                   "train_raw_slice (INFERENCE)": X_raw.shape},
        "dtypes": {"X_train_smote": str(X_train_smote.dtype), "y_train_smote": str(y_train_smote.dtype), "X_test": str(X_test.dtype), "y_test": str(y_test.dtype)},
        "train_is_smote": True,
        "raw_csv_available": RAW_CSV is not None,
        "class_distribution": {"test": dist(y_test), "train_raw_slice": dist_raw, "train_smote": dist_smote},
        "class_names": CLASS_NAMES,
    },
    "data_range": {r["set"]: {k: r[k] for k in ("min", "max", "mean", "std", "NaN", "Inf")} for r in stats_df.to_dict("records")},
    "sequence_length": {"config_sequence_len": CFG.sequence_len, "used_in_code": SEQ_LEN_USED_IN_CODE,
                        "trans_dim": CFG.trans_dim, "final_len_186": T186["final_len"], "final_len_187": T187["final_len"],
                        "compatible_input_lengths": [min(ok_L), max(ok_L)]},
    "model": {"input_shape": [AUDIT_BATCH, 1, SEQ_LEN], "output_shape": list(T["out"]), "n_params": n_params,
              "n_params_unused_encoder_layer": n_unused_params, "modified": False},
    "class_weights_cal": {"train_raw": W_RAW, "train_smote": W_SMOTE, "baseline_resnet18": BASELINE_CLASS_WEIGHTS},
    "forward_status": "PASS" if FORWARD_OK else "FAIL",
    "loss_status": "PASS" if LOSS_OK else "FAIL",
    "loss_value_random_init_batch8": float(loss.item()),
    "backward_status": "PASS" if BACKWARD_OK else "FAIL",
    "test_integrity": "PASS" if TEST_UNCHANGED else "FAIL",
    "repo_integrity": "PASS" if REPO_CLEAN else "FAIL",
    "audit_table": [{"no": n, **a} for n, a in sorted(AUDIT.items())],
    "adaptation_notes": ADAPTATIONS,
    "residual_unknowns": UNKNOWNS,
    "final_feasibility": VERDICT,
    "recommendation": RECOMMENDATION,
}

if SAVE_JSON:
    os.makedirs(RESULTS_DIR, exist_ok=True)
    out_path = os.path.join(RESULTS_DIR, "ecgtransform_audit.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False, default=_jd)
    print("Saved:", out_path)
else:
    print("SAVE_JSON = False -> tidak ada file ditulis.")

Saved: /home/claude/work/results_test/ecgtransform_audit.json
